# 10 — Biological results

RNA-seq measures **target labels**, not predictor features. These analyses describe RNA expression, not tumor surface protein abundance or immunotherapy response.

In [ ]:
from pathlib import Path
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

def find_root():
    for p in [Path.cwd(),*Path.cwd().parents]:
        if (p/"config"/"genes.json").exists(): return p
    raise FileNotFoundError("Project root not found")
ROOT=find_root()
genes=json.loads((ROOT/"config"/"genes.json").read_text())["taa_target_genes"]
cols=[f"{g}_rna_log2tpm" for g in genes]
df=pd.read_csv(ROOT/"data"/"processed"/"modeling_table.csv")
assert set(cols+["project_id"]).issubset(df.columns)
cohorts=["TCGA-BRCA","TCGA-OV","TCGA-UCEC","TCGA-LUAD","TCGA-LUSC","TCGA-BLCA","TCGA-HNSC","TCGA-STAD","TCGA-COAD","TCGA-READ"]
OUT=ROOT/"reports"/"figures"/"biology"
OUT.mkdir(parents=True,exist_ok=True)
def export(fig,name):
    fig.savefig(OUT/f"{name}.png",dpi=300,bbox_inches="tight")
    fig.savefig(OUT/f"{name}.pdf",bbox_inches="tight")
    plt.show()


## 1. Mean and relative TAA RNA expression

Each heatmap uses mean cohort log2(TPM+1); the relative heatmap standardizes each gene across ten cohorts.

In [ ]:
means=df.groupby("project_id")[cols].mean().reindex(cohorts)
means.columns=genes
fig,ax=plt.subplots(figsize=(13,7))
sns.heatmap(means,annot=True,fmt=".2f",cmap="viridis",ax=ax,cbar_kws={"label":"Mean log2(TPM + 1)"})
ax.set(xlabel="Tumor-associated antigen gene",ylabel="TCGA cancer cohort",title="Mean TAA RNA expression by cancer type")
plt.setp(ax.get_xticklabels(),rotation=40,ha="right")
export(fig,"mean_taa_expression")
relative=means.apply(lambda s:(s-s.mean())/s.std(ddof=0) if s.std(ddof=0)>0 else s*0,axis=0)
fig,ax=plt.subplots(figsize=(13,7))
sns.heatmap(relative,annot=True,fmt=".2f",cmap="vlag",center=0,ax=ax,cbar_kws={"label":"Within-gene relative z-score"})
ax.set(xlabel="Tumor-associated antigen gene",ylabel="TCGA cancer cohort",title="Relative TAA RNA enrichment by cancer type")
plt.setp(ax.get_xticklabels(),rotation=40,ha="right")
export(fig,"relative_taa_expression")
means.to_csv(OUT/"mean_taa_expression.csv")
relative.to_csv(OUT/"relative_taa_expression.csv")

## 2. Between- and within-cohort heterogeneity

Boxes show median and IQR, with whiskers representing nonoutlier observations.

In [ ]:
selected=["ERBB2","FOLR1","MUC16","EPCAM","CD276","TACSTD2"]
long=df[["project_id"]+[f"{g}_rna_log2tpm" for g in selected]].melt(id_vars="project_id",var_name="gene",value_name="expression")
long["gene"]=long["gene"].str.replace("_rna_log2tpm","",regex=False)
fig,axes=plt.subplots(3,2,figsize=(15,16),sharex=True)
for ax,g in zip(axes.flat,selected):
    sns.boxplot(data=long[long.gene==g],x="project_id",y="expression",order=cohorts,showfliers=False,ax=ax)
    ax.set(title=g,xlabel="",ylabel="RNA expression log2(TPM + 1)")
    ax.tick_params(axis="x",rotation=45)
    plt.setp(ax.get_xticklabels(),ha="right",fontsize=8)
fig.suptitle("TAA RNA expression varies within and across tumor cohorts",fontsize=15)
fig.tight_layout()
export(fig,"taa_expression_heterogeneity")

## 3. Feature-group ablation

Predictor-group comparisons assess held-out association, not causation. The code to retrain these models is in notebook 09.

In [ ]:
path=ROOT/"results"/"metrics"/"feature_group_ablation_metrics.csv"
if path.exists():
    abl=pd.read_csv(path)
    if "gene" not in abl.columns:
        abl["gene"]=abl["target"].str.replace("_rna_log2tpm","",regex=False)
    names=["Clinical/cancer type only","Copy-number only","Mutation only","Copy-number + mutation","All features"]
    matrix=abl.pivot_table(index="gene",columns="feature_group",values="r2").reindex(index=genes,columns=names)
    fig,ax=plt.subplots(figsize=(12,7))
    sns.heatmap(matrix,annot=True,fmt=".2f",cmap="viridis",ax=ax,cbar_kws={"label":"Held-out test R²"})
    ax.set(title="Feature-group ablation: non-RNA predictors of TAA expression",ylabel="TAA gene",xlabel="Input feature group")
    plt.setp(ax.get_xticklabels(),rotation=32,ha="right")
    export(fig,"feature_group_ablation")
else:
    print("No ablation results: execute 09_Feature_Group_Ablation.ipynb first.")

## 4. Predicted versus observed on held-out test tumors

Each dot is an actual–predicted pair. The dashed line denotes ideal predictions; blue dots do not represent a distinct ‘actual’ group.

In [ ]:
pred_dir=ROOT/"results"/"predictions"
preferred=["exp02_mlp_dropout_40_predictions.csv","exp02_mlp_dropout_20_predictions.csv","exp01_standard_mlp_predictions.csv"]
candidates=[pred_dir/p for p in preferred if (pred_dir/p).exists()] or sorted(pred_dir.glob("*_predictions.csv"))
if candidates:
    f=candidates[0]
    pred=pd.read_csv(f)
    fig,axes=plt.subplots(2,2,figsize=(12,10))
    for ax,g in zip(axes.flat,["EPCAM","FOLR1","ERBB2","CD276"]):
        t=f"{g}_rna_log2tpm"
        actual=next((x for x in [f"{t}_true",f"{g}_true"] if x in pred.columns),None)
        estimated=next((x for x in [f"{t}_pred",f"{t}_pred_mlp",f"{g}_pred"] if x in pred.columns),None)
        if actual is None or estimated is None:
            raise KeyError(f"Could not match true/prediction for {g} in {f.name}")
        d=pred[[actual,estimated]].apply(pd.to_numeric,errors="coerce").dropna()
        if len(d)<2: raise ValueError(f"Too few predictions for {g}")
        ax.scatter(d[actual],d[estimated],alpha=.5,s=20,label="Tumor samples")
        low=min(d.min()); high=max(d.max())
        ax.plot([low,high],[low,high],"--",c="black",label="Perfect prediction")
        ax.set(xlabel="Observed log2(TPM + 1)",ylabel="Predicted log2(TPM + 1)",title=f"{g} — test R² {r2_score(d[actual],d[estimated]):.2f}")
        ax.legend(loc="lower right",fontsize=8)
    fig.suptitle(f"Predicted vs observed TAA RNA expression ({f.stem})")
    fig.tight_layout()
    export(fig,"predicted_vs_observed_taas")
else:
    print("No prediction CSV available; run architecture notebooks first.")

## Interpretation

Cancer-type enrichment and heterogeneity suggest antigen- and cohort-specific expression. Some expression patterns can be predicted from DNA and clinical features, but RNA expression does not establish surface protein abundance, therapeutic targetability, or clinical benefit.